# 03회차 실습 노트북 — 데이터 분석을 위한 준비 — 수집 계획

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/03_collection_plan.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/03_collection_plan.html](https://adminmrobo.github.io/fintech_data_analysis/03_collection_plan.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 데이터 사전: 열 이름만 보고는 뜻을 모른다

In [2]:
ld = load('Revolution4/loan_data_description.csv')
ld = ld[ld.LoanStatNew.notna()]
print(len(ld), '개 열 설명'); ld.head(10)

78 개 열 설명


,Unnamed: 0,LoanStatNew,Description
0,0,addr_state,The state provided by the borrower in the loan...
1,1,annual_inc,The self-reported annual income provided by th...
2,2,annual_inc_joint,The combined self-reported annual income provi...
3,3,application_type,Indicates whether the loan is an individual ap...
4,4,collection_recovery_fee,post charge off collection fee
5,5,collections_12_mths_ex_med,Number of collections in 12 months excluding m...
6,6,delinq_2yrs,The number of 30+ days past-due incidences of ...
7,7,desc,Loan description provided by the borrower
8,8,dti,A ratio calculated using the borrower’s total ...
9,9,dti_joint,A ratio calculated using the co-borrowers' tot...


## 2. 대결 재현: 수집 주기가 길면 숫자가 얼마나 낡나
주가를 매일 모으면 오늘 값을 바로 쓰고, 한 달(21거래일)마다 모으면 그 사이에는 마지막으로 받은 값을 씁니다. 그 차이를 평균 상대 오차로 봅니다.

In [3]:
p = load('Revolution3/portfolio_data.csv')
def stale(x, gap):
    x = np.asarray(x, float)
    est = np.array([x[(i // gap) * gap - 1] if (i // gap) * gap >= 1 else x[0] for i in range(len(x))])
    m = np.arange(len(x)) >= gap
    return np.mean(np.abs(x[m] - est[m]) / x[m])
for c in ['AMZN', 'BTC']:
    print(c, {g: f'{stale(p[c], gap):.1%}' for g, gap in [('매일', 1), ('주 1회', 5), ('월 1회', 21), ('분기 1회', 63)]})

AMZN {'매일': '1.3%', '주 1회': '2.3%', '월 1회': '4.4%', '분기 1회': '6.1%'}
BTC {'매일': '3.4%', '주 1회': '6.6%', '월 1회': '13.3%', '분기 1회': '27.9%'}


**AI 판단 연습** — "매일 모으면 오차 0%"는 당연한 결과일까요? 매일 모으는 데 드는 비용(저장·점검·사람)까지 생각하면 어떤 데이터는 월 1회로도 충분할 수 있어요. 어떤 데이터일까요?